# Day 1 · Hands-on 6 — Choosing a test
**About 30 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


Five short research scenarios. For each one:
1. Read the card.
2. Paste the **AI prompt** into Claude / ChatGPT and note the test it recommends.
3. Check its answer against the decision guide below.
4. Set `my_test` in the code cell and run it. Then try the alternatives and compare.



> **Two AI roles today — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to change, fix or explain *code*.
> - 🎓 **Claude / ChatGPT (other tab) = tutor you must check.** Ask it to *interpret*, then score its answer: **correct · complete · specific · actionable**.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## 🗺️ Decision guide

| Your question | Typical choice | Key assumptions |
|---|---|---|
| Two **independent** groups, continuous outcome, roughly symmetric or large samples | **Welch t-test** | Independent observations; mean is a sensible summary |
| Two independent groups, **small samples, skewed data or outliers** | **Mann–Whitney** (or t-test on log scale) | Independent observations; compares ranks, not means |
| **Paired** measurements (same subject before/after, matched samples) | **Paired t-test** (or Wilcoxon signed-rank) | Pairs independent of each other; differences roughly symmetric |
| Two **proportions** (yes/no outcome) | **Chi-square test**; **Fisher's exact test** when expected counts are small (< 5) | Independent observations |
| **Time until an event**, with some patients not yet having had it (censored) | **Log-rank test / Cox regression** (Day 3) | Censoring unrelated to outcome |

**Rule of thumb:** the design chooses the test *before* you see any p-value. Choosing the test that gives the nicest p-value is a form of p-hacking.

---
## Card 1 · Two groups
Do pre- and postmenopausal patients in GBSG2 differ in tumour size? (290 vs 396 patients.)

**🎓 AI prompt:** *I want to compare tumour size (mm) between 290 premenopausal and 396 postmenopausal breast cancer patients. Which statistical test should I use, and what are its assumptions?*

In [ ]:
# 🔧 CHANGE THIS: your choice — "welch_t", "mann_whitney" or "paired_t"
my_test = "welch_t"

pre = df.loc[df.menostat == "Pre", "tsize"]
post = df.loc[df.menostat == "Post", "tsize"]
df.boxplot(column="tsize", by="menostat", figsize=(5, 3.5)); plt.suptitle(""); plt.show()

if my_test == "welch_t":
    print(f"Welch t-test: p = {stats.ttest_ind(pre, post, equal_var=False).pvalue:.3f}")
elif my_test == "mann_whitney":
    print(f"Mann–Whitney: p = {stats.mannwhitneyu(pre, post).pvalue:.3f}")
elif my_test == "paired_t":
    print("❌ Not possible: these are different patients, so there is nothing to pair.")

---
## Card 2 · Paired data
Tumour volume (mm³) in 10 mice, measured **before and after** two weeks of treatment.

**🎓 AI prompt:** *I have tumour volumes for a "before" and an "after" group: before = 645, 93, 450, 332, 346, 374, 158, 372, 296, 799; after = 552, 74, 378, 272, 278, 312, 141, 313, 266, 676. Which statistical test should I use to see if the treatment reduced tumour volume?*

(The prompt calls them "groups", which is deliberate. Did the AI notice they are the same mice?)

In [ ]:
# 🔧 CHANGE THIS: your choice — "welch_t", "paired_t" or "wilcoxon"
my_test = "paired_t"

before = np.array([645, 93, 450, 332, 346, 374, 158, 372, 296, 799])
after  = np.array([552, 74, 378, 272, 278, 312, 141, 313, 266, 676])

plt.figure(figsize=(4, 3.5))
for b, a in zip(before, after):
    plt.plot([0, 1], [b, a], "o-", color="gray")
plt.xticks([0, 1], ["before", "after"]); plt.ylabel("Tumour volume (mm³)"); plt.show()

if my_test == "welch_t":
    print(f"Welch t-test (treats before/after as unrelated groups): p = {stats.ttest_ind(before, after, equal_var=False).pvalue:.3f}")
elif my_test == "paired_t":
    print(f"Paired t-test: p = {stats.ttest_rel(before, after).pvalue:.4f}")
elif my_test == "wilcoxon":
    print(f"Wilcoxon signed-rank: p = {stats.wilcoxon(before, after).pvalue:.4f}")
print(f"Every mouse changed by: {(after - before).tolist()}")

---
## Card 3 · Proportions, small numbers
Tumours developed in **1 of 10** mice on a protective diet and **6 of 10** control mice.

**🎓 AI prompt:** *In my experiment, 1 of 10 mice on a protective diet developed tumours versus 6 of 10 control mice. Which statistical test should I use to compare the proportions?*

In [ ]:
# 🔧 CHANGE THIS: your choice — "chi_square" or "fisher"
my_test = "fisher"

#                 tumour  no tumour
table = np.array([[1, 9],     # protective diet
                  [6, 4]])    # control
expected = stats.chi2_contingency(table)[3]
print("Expected counts if diet had no effect:")
print(expected.round(1), "\n")

if my_test == "chi_square":
    print(f"Chi-square test: p = {stats.chi2_contingency(table, correction=False)[1]:.3f}")
elif my_test == "fisher":
    print(f"Fisher's exact test: p = {stats.fisher_exact(table).pvalue:.3f}")

---
## Card 4 · Non-normal data
Serum IL-6 (pg/mL) in 8 healthy controls and 8 patients. Cytokine levels are typically skewed, and one patient has a very high value.

**🎓 AI prompt:** *I measured IL-6 in 8 controls (5.5, 3.8, 4.1, 1.5, 12.3, 8.9, 4.2, 7.4) and 8 patients (82.9, 9.1, 19.6, 10.3, 10.2, 8.1, 15.1, 11.4). Which test should I use to compare the groups?*

In [ ]:
# 🔧 CHANGE THIS: your choice — "welch_t", "mann_whitney" or "welch_t_log"
my_test = "mann_whitney"

controls = np.array([5.5, 3.8, 4.1, 1.5, 12.3, 8.9, 4.2, 7.4])
patients = np.array([82.9, 9.1, 19.6, 10.3, 10.2, 8.1, 15.1, 11.4])

plt.figure(figsize=(4, 3.5))
plt.plot(np.zeros(8), controls, "o", alpha=0.7); plt.plot(np.ones(8), patients, "o", alpha=0.7)
plt.xticks([0, 1], ["controls", "patients"]); plt.ylabel("IL-6 (pg/mL)"); plt.show()

if my_test == "welch_t":
    print(f"Welch t-test: p = {stats.ttest_ind(controls, patients, equal_var=False).pvalue:.3f}")
elif my_test == "mann_whitney":
    print(f"Mann–Whitney: p = {stats.mannwhitneyu(controls, patients).pvalue:.4f}")
elif my_test == "welch_t_log":
    print(f"Welch t-test on log(IL-6): p = {stats.ttest_ind(np.log(controls), np.log(patients), equal_var=False).pvalue:.4f}")

---
## Card 5 · The trap
Do GBSG2 patients who received hormonal therapy stay **recurrence-free longer**? `time` is days from surgery until recurrence *or until the last follow-up visit*; `cens` = 1 if a recurrence was seen, 0 if the patient was still recurrence-free when follow-up ended.

**🎓 AI prompt:** *I want to compare the column "time" (days until recurrence) between patients with and without hormonal therapy in the GBSG2 dataset. Which test should I use?*

In [ ]:
# 🔧 CHANGE THIS: your choice — "welch_t", "mann_whitney" or "logrank"
my_test = "welch_t"

from lifelines.statistics import logrank_test
yes, no = df[df.horTh == "yes"], df[df.horTh == "no"]
print(f"Patients still recurrence-free at last follow-up (censored): {(df.cens == 0).mean():.0%}\n")

if my_test in ("welch_t", "mann_whitney"):
    if my_test == "welch_t":
        p = stats.ttest_ind(yes.time, no.time, equal_var=False).pvalue
    else:
        p = stats.mannwhitneyu(yes.time, no.time).pvalue
    print(f"{my_test}: p = {p:.3f}")
    print("⚠️ This treats 'last seen recurrence-free at day 1,500' as if the patient recurred at day 1,500.")
    print("   For censored patients the true time is LONGER than recorded, so this test answers the wrong question.")
elif my_test == "logrank":
    res = logrank_test(yes.time, no.time, event_observed_A=yes.cens, event_observed_B=no.cens)
    print(f"Log-rank test (uses censoring correctly): p = {res.p_value:.4f}")
    print("We'll unpack this test and Kaplan–Meier curves on Day 3.")

## 🧮 Gemini in Colab (calculator)
Select the Card 4 cell and ask:
> *Add a check of normality for each group before the test.*

Then discuss: with only 8 values per group, how much can a normality test tell you? Is a plot more useful?

## 💬 Discuss with your partner
- In which cards did the AI pick the test you would pick? Where did it go wrong, and what in the prompt misled it?
- Cards 2, 3 and 4: the alternatives gave quite different p-values. Why is it still wrong to choose the test *after* seeing them?
- Card 5: even with the right test, Hands-on 3 showed the therapy groups differ in age and menopausal status. What does that mean for interpreting the log-rank result?